# 04 — Small CNN trained from scratch: RGB vs 13 spectral bands

This notebook trains a compact convolutional neural network from scratch on RGB (3 channels) and on all 13 bands, using the **same fixed split, training-only normalisation and augmentation** from notebook 02. The only architecture change is the number of input channels.

**Functions:** `src/piece_of_crop/cnn.py` (`SmallCNN`, `build_cnn`) and `src/piece_of_crop/train.py` (`set_seed`, `fit`, `evaluate_model`, `predictions_table`). The same training loop is used for the ResNet in notebook 05.

**Inputs:** `results/split.csv`, `results/band_stats.csv` and the shared EuroSAT cache (`data/eurosat_cache.npy`) or raw `data/EuroSAT_MS` files.

**Outputs:** `results/cnn_results.csv`, `results/cnn_history.csv`, test prediction CSVs `results/predictions/cnn_<input>_seed<n>.csv`, training curves in `results/figures/` and local model weights in `models/` (do not commit weights or data).

**Design:** Conv–BatchNorm–ReLU blocks and pooling, followed by global average pooling and a 10-class head. Train on three seeds, choose each best checkpoint using *validation loss only*, and evaluate the held-out test split afterward. No hyperparameter tuning on test data.

## 1. Shared setup and data from notebook 02

All functions come from the installed `piece_of_crop` package. `find_project_root` finds the project folder whether the notebook runs from the project root or from `notebooks/`.

In [ ]:
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.metrics import classification_report

from piece_of_crop import (CLASS_NAMES, build_cnn, check_test_order, count_parameters, evaluate_model,
                           find_project_root, fit, get_cache, get_device, make_loaders, predictions_table,
                           read_split, set_seed)

SEEDS = (0, 1, 2)
BATCH_SIZE = 64
MAX_EPOCHS = 10
PATIENCE = 3
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4

project_root = find_project_root()
data_dir = project_root / "data" / "EuroSAT_MS"
results_dir = project_root / "results"
figures_dir = results_dir / "figures"
predictions_dir = results_dir / "predictions"
models_dir = project_root / "models"
for folder in (results_dir, figures_dir, predictions_dir, models_dir):
    folder.mkdir(parents=True, exist_ok=True)

split = read_split(results_dir / "split.csv")
band_stats = pd.read_csv(results_dir / "band_stats.csv")
cache = get_cache(split, data_dir, project_root / "data" / "eurosat_cache.npy")
label_ids = np.arange(len(CLASS_NAMES))
assert np.array_equal(np.sort(split["label"].unique()), label_ids), "Expected labels 0..9."
device = get_device()

print("Project root:", project_root)
print("Device:", device)
print("Patches:", split["split"].value_counts().to_dict())
print("Cache:", cache.shape)

## 2. Model architecture

A deliberately small CNN keeps the RGB and 13-band comparison interpretable. It uses three Conv–BatchNorm–ReLU–MaxPool blocks (32, 64, 128 channels), global average pooling and a linear classification layer. Inputs stay at the original 64 × 64 resolution; no pretrained weights are used.

The class is `SmallCNN` in `src/piece_of_crop/cnn.py`. The cell below only checks the output shape and counts the parameters.

In [ ]:
for input_type, channels in (("rgb", 3), ("13band", 13)):
    test_model = build_cnn(input_type)
    assert test_model(torch.zeros(2, channels, 64, 64)).shape == (2, 10)
    print(f"{input_type}: {count_parameters(test_model):,} trainable parameters")
print(test_model)

## 3. Training and evaluation helpers

Use cross-entropy loss, AdamW and early stopping based on **validation loss**. The test split is only evaluated after selecting the best validation checkpoint. Accuracy, macro-F1 and per-class classification are calculated with the same definitions as notebook 03.

The loop is `fit` in `src/piece_of_crop/train.py`: one epoch of training (`train_epoch`), then `evaluate_model` on the validation set, early stopping when the validation loss has not improved for `PATIENCE` epochs, and the best epoch is restored at the end.

## 4. Train RGB and multispectral CNNs

For every seed, recreate the data loaders so the training augmentations follow that seed. Keep identical optimiser settings and epoch limits for RGB and 13-band. Models and predictions are saved separately; the output CSV format matches notebook 03. On an M3 Mac, the notebook will use MPS if PyTorch supports it. The full experiment trains six networks and may be computationally expensive.

In [ ]:
history_rows = []
result_rows = []

for input_type, n_channels in (("rgb", 3), ("13band", 13)):
    for seed in SEEDS:
        print(f"\n=== CNN {input_type}, seed {seed} ===")
        set_seed(seed)
        loaders = make_loaders(split, cache, band_stats, input_type, batch_size=BATCH_SIZE, seed=seed)
        sample_images, sample_labels = next(iter(loaders["train"]))
        assert sample_images.shape[1:] == (n_channels, 64, 64)

        model = build_cnn(input_type, num_classes=len(CLASS_NAMES)).to(device)
        optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
        history, loss_fn = fit(model, loaders, optimizer, device, MAX_EPOCHS, PATIENCE,
                               run_info={"model": "SmallCNN", "input": input_type, "seed": seed})
        history_rows.extend(history)

        # Best checkpoint selected on the validation set; the test set is used only now.
        torch.save({"state_dict": model.cpu().state_dict(), "input": input_type,
                    "seed": seed, "num_classes": len(CLASS_NAMES)},
                   models_dir / f"cnn_{input_type}_seed{seed}.pt")
        model.to(device)

        for subset in ("val", "test"):
            outcome = evaluate_model(model, loaders[subset], loss_fn, device)
            result_rows.append({
                "model": "SmallCNN", "input": input_type, "seed": seed, "split": subset,
                "accuracy": outcome["accuracy"], "macro_f1": outcome["macro_f1"],
                "loss": outcome["loss"], "n_samples": len(outcome["y_true"]),
            })
            if subset == "test":
                check_test_order(split, outcome["y_true"])
                test_files = split.loc[split["split"].eq("test"), "file"]
                predictions_table(test_files, outcome["y_true"], outcome["proba"]).to_csv(
                    predictions_dir / f"cnn_{input_type}_seed{seed}.csv", index=False)

cnn_history = pd.DataFrame(history_rows)
cnn_results = pd.DataFrame(result_rows)
cnn_history.to_csv(results_dir / "cnn_history.csv", index=False)
cnn_results.to_csv(results_dir / "cnn_results.csv", index=False)
display(cnn_results)

## 5. Training curves and model comparison

Inspect training vs validation loss, the mean and standard deviation of accuracy/macro-F1 over seeds, and per-class classification reports. The plot and tables are based on executed model results; they are not pre-filled estimates.

In [ ]:
import matplotlib.pyplot as plt

for input_type in ('rgb', '13band'):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for seed in SEEDS:
        run = cnn_history[(cnn_history['input'] == input_type) & (cnn_history['seed'] == seed)]
        axes[0].plot(run['epoch'], run['train_loss'], ls='--', alpha=.6, label=f'train seed {seed}')
        axes[0].plot(run['epoch'], run['val_loss'], label=f'val seed {seed}')
        axes[1].plot(run['epoch'], run['val_accuracy'], label=f'seed {seed}')
    axes[0].set(title=f'{input_type}: loss curves', xlabel='Epoch', ylabel='Cross-entropy loss')
    axes[1].set(title=f'{input_type}: validation accuracy', xlabel='Epoch', ylabel='Accuracy', ylim=(0, 1))
    for ax in axes:
        ax.grid(alpha=.3)
        ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(figures_dir / f'cnn_{input_type}_training.png', dpi=150, bbox_inches='tight')
    plt.show()

print('CNN metrics, by input and split (mean and std across seeds):')
display(cnn_results.groupby(['split', 'input'])[['accuracy', 'macro_f1']].agg(['mean', 'std']).round(4))

for input_type in ('rgb', '13band'):
    pred = pd.read_csv(predictions_dir / f'cnn_{input_type}_seed0.csv')
    print(f'\nPer-class report: {input_type}, seed 0')
    print(classification_report(pred['y_true'], pred['y_pred'],
                                labels=label_ids, target_names=list(CLASS_NAMES), zero_division=0))

print('Saved metrics to:', results_dir / 'cnn_results.csv')
print('Saved predictions to:', predictions_dir)
print('Saved training curves to:', figures_dir)